## Short EDA of the Account List from University of Innsbruck

In [1]:
import pandas as pd

In [2]:
df_accs = pd.read_csv("10910_da_de_v1_0.csv")

df_accs.head()

,version,doi,id_var,Typ,Nachname,Vorname,Gender,Geburtsjahr,Partei,Region,...,V10_Kommentare,V10_Shares,V10_Setting,V10_Selfie,V10_Kleidung,V10_Call,V10_Musik,V10_Familie,V10_Partei,V10_Trend
0,1.0 (2026-03-05),doi:10.11587/BA1ORN,1,Person,Abrahamczik,Nina,weiblich,1982.0,SPÖ,Wien,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0 (2026-03-05),doi:10.11587/BA1ORN,2,Person,Abwerzger,Markus,männlich,1975.0,FPÖ,Tirol,...,152.0,107.0,Arbeit,nein,leger,nein,nein,nein,erkennbar,nein
2,1.0 (2026-03-05),doi:10.11587/BA1ORN,3,Person,Achhorner,Evelyn,weiblich,1965.0,FPÖ,Tirol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0 (2026-03-05),doi:10.11587/BA1ORN,4,Person,Achleitner,Markus,männlich,1969.0,ÖVP,Oberösterreich,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0 (2026-03-05),doi:10.11587/BA1ORN,5,Person,Ackerl,Alexander,männlich,1991.0,SPÖ,Wien,...,27.0,23.0,Arbeit,nein,formal,nein,nein,nein,erkennbar,nein


In [3]:
print(df_accs.columns)

Index(['version', 'doi', 'id_var', 'Typ', 'Nachname', 'Vorname', 'Gender',
       'Geburtsjahr', 'Partei', 'Region',
       ...
       'V10_Kommentare', 'V10_Shares', 'V10_Setting', 'V10_Selfie',
       'V10_Kleidung', 'V10_Call', 'V10_Musik', 'V10_Familie', 'V10_Partei',
       'V10_Trend'],
      dtype='str', length=150)


### count the number of tiktok accounts

In [4]:
print(f"Number of TikTok accouns: {(df_accs["TikTok"] == "ja").sum()}")

Number of TikTok accouns: 238


## count the number of TikTok accounts per Party

In [5]:
only_accs = df_accs[df_accs["TikTok"]=="ja"].copy()

only_accs["Partei"].value_counts()

Partei
SPÖ          61
FPÖ          51
ÖVP          48
GRÜNE        42
NEOS         27
MFG           3
KPÖ PLUS      2
KPÖ           2
parteilos     1
FRITZ         1
Name: count, dtype: int64

## Prepare a df containing all the accounts for scraping (only the necessary columns)

In [6]:
df_scrape = only_accs[["id_var", "Typ", "Nachname", "Vorname", "Gender", "Geburtsjahr", "Partei", "Region", "Funktion", "TikTok_Link"]].copy()

df_scrape.head()

,id_var,Typ,Nachname,Vorname,Gender,Geburtsjahr,Partei,Region,Funktion,TikTok_Link
1,2,Person,Abwerzger,Markus,männlich,1975.0,FPÖ,Tirol,Landtag,https://www.tiktok.com/@markusabwerzger
4,5,Person,Ackerl,Alexander,männlich,1991.0,SPÖ,Wien,Landtag,https://www.tiktok.com/@alexander.ackerl
6,7,Person,Aigner,Joachim,männlich,1976.0,MFG,Oberösterreich,Landtag,https://www.tiktok.com/@joachimaigner27
16,17,Person,Angerer,Erwin,männlich,1964.0,FPÖ,Kärnten,Landtag,https://www.tiktok.com/@erwin_angerer
19,20,Person,Antlinger,Thomas,männlich,1994.0,SPÖ,Oberösterreich,Landtag,https://www.tiktok.com/@thantlinger


### rename the columns

In [7]:
df_scrape = df_scrape.rename(columns={
    "id_var": "id",
    "Typ": "type",
    "Nachname": "last_name",
    "Vorname": "first_name",
    "Gender": "gender",
    "Geburtsjahr": "birth_year",
    "Partei": "party",
    "Region": "region",
    "Funktion": "position",
    "TikTok_Link": "tiktok_link"
})

df_scrape.head()

,id,type,last_name,first_name,gender,birth_year,party,region,position,tiktok_link
1,2,Person,Abwerzger,Markus,männlich,1975.0,FPÖ,Tirol,Landtag,https://www.tiktok.com/@markusabwerzger
4,5,Person,Ackerl,Alexander,männlich,1991.0,SPÖ,Wien,Landtag,https://www.tiktok.com/@alexander.ackerl
6,7,Person,Aigner,Joachim,männlich,1976.0,MFG,Oberösterreich,Landtag,https://www.tiktok.com/@joachimaigner27
16,17,Person,Angerer,Erwin,männlich,1964.0,FPÖ,Kärnten,Landtag,https://www.tiktok.com/@erwin_angerer
19,20,Person,Antlinger,Thomas,männlich,1994.0,SPÖ,Oberösterreich,Landtag,https://www.tiktok.com/@thantlinger


### make the gender english and replace tiktok link with tiktok username

In [8]:
df_scrape["gender"] = df_scrape["gender"].replace({
    "männlich": "male",
    "weiblich": "female"
})

df_scrape["tiktok_username"] = df_scrape["tiktok_link"].apply(lambda x: x.split("@", 1)[1].split("?", 1)[0])#take everything after "@" and before "?"

df_scrape = df_scrape.drop(columns="tiktok_link")

df_scrape["birth_year"] = df_scrape["birth_year"].fillna(-1)#missing birth year is encoded as -1 so that we can use int type
df_scrape["birth_year"] = df_scrape["birth_year"].astype(int)

df_scrape.head()

,id,type,last_name,first_name,gender,birth_year,party,region,position,tiktok_username
1,2,Person,Abwerzger,Markus,male,1975,FPÖ,Tirol,Landtag,markusabwerzger
4,5,Person,Ackerl,Alexander,male,1991,SPÖ,Wien,Landtag,alexander.ackerl
6,7,Person,Aigner,Joachim,male,1976,MFG,Oberösterreich,Landtag,joachimaigner27
16,17,Person,Angerer,Erwin,male,1964,FPÖ,Kärnten,Landtag,erwin_angerer
19,20,Person,Antlinger,Thomas,male,1994,SPÖ,Oberösterreich,Landtag,thantlinger


In [9]:
df_scrape.to_csv("austriaPoliticians.csv", index=False)

## Filter 